In [22]:
from pathlib import Path
import csv
import soundfile as sf
from IPython.display import Audio, display

In [12]:
DATA_DIR = Path("datasets/ssw/data")
ANN_PATH = Path("datasets/ssw/annotations.csv")
OUT_DIR = Path("datasets/ssw_clips")  # or DATA_DIR
N_FILES = 10
CLIP_SEC = 5.0

In [19]:
def load_annotations_by_file(path: Path) -> dict[str, list[tuple[float, float]]]:
    """All annotations for each filename (appearance order). 
    Return list of start time and end time for each file.
    """
    by_file: dict[str, list[tuple[float, float]]] = {}
    with path.open(newline="") as f:
        for row in csv.DictReader(f):
            filename = row["Filename"]
            if filename not in by_file:
                by_file[filename] = []
            by_file[filename].append((
                float(row["Start Time (s)"]),
                float(row["End Time (s)"]),
            ))    
    return by_file

def overlaps(a0, a1, b0, b1) -> bool:
    """Check if two ranges overlap"""
    return a0 < b1 and b0 < a1  

def pick_signal_and_noise_timerange(
    intervals: list[tuple[float, float]],
    file_duration_sec: float,
    clip_sec: float = 5.0,
    noise_target_gap: float = 1.0
) -> tuple[float, float, float, float] | None:
    """
    First annotation with (end-start) >= clip_sec, whose [start-clip_sec-noise_target_gap, start]
    or [start,end+clip_sec+noise_target_gap] does not overlap any other annotation.
    Assume noise_target_gap between target sound and noise.
    Annotations are sorted.

    Returns (sig0, sig1, noise0, noise1) or None.
    """
    for i, (start, end) in enumerate(intervals) : 
        if end - start < clip_sec:
            continue # too short 
        
        prev_end = intervals[i - 1][1] if i > 0 else 0.0
        next_start = intervals[i + 1][0] if i + 1 < len(intervals) else file_duration_sec

        # prefer noise right before the match 
        if start - prev_end >= clip_sec + noise_target_gap:
            noise0, noise1 = start - clip_sec - noise_target_gap, start - noise_target_gap
            return start, start + clip_sec, noise0, noise1
        
        # else noise right after the match
        if next_start - end >= clip_sec + noise_target_gap:
            noise0, noise1 = end + noise_target_gap, end + clip_sec + noise_target_gap
            return start, start + clip_sec, noise0, noise1
    
    return None

def load_first_annotation_per_file(path: Path) -> dict[str, tuple[float, float]]:
    """First CSV row for each filename (appearance order)."""
    first = {}
    with path.open(newline="") as f:
        for row in csv.DictReader(f):
            fn = row["Filename"]
            if fn in first:
                continue
            first[fn] = (
                float(row["Start Time (s)"]),
                float(row["End Time (s)"]),
            )
    return first

def read_and_save_slice(in_path, t0, t1, out_path: Path):
    info = sf.info(in_path)
    sr = info.samplerate
    start = int(t0 * sr)
    frames =int((t1 - t0) * sr)
    audio, _ = sf.read(in_path, start=start, frames=frames, dtype='float32')
    sf.write(out_path, audio, sr)

In [20]:
annotations = load_annotations_by_file(ANN_PATH)

OUT_DIR.mkdir(parents=True, exist_ok=True)
files = sorted(DATA_DIR.glob("*.flac"))[:N_FILES]

for path in files:
    intervals = annotations[path.name]
    if not intervals:
        print(f"Skip {path.name} (no annotations)")
        continue

    info = sf.info(path)
    duration = info.duration
    sr = info.samplerate
        
    picked = pick_signal_and_noise_timerange(
        intervals=intervals,
        file_duration_sec=duration,
        clip_sec=CLIP_SEC,
    )
    if picked is None:
        print(f"Skip {path.name} (no picked)")
        continue

    sig0, sig1, noise0, noise1 = picked
    stem = path.stem

    read_and_save_slice(path, sig0, sig1, OUT_DIR / f"{stem}_signal.flac")
    read_and_save_slice(path, noise0, noise1, OUT_DIR / f"{stem}_noise.flac")
    print(f"Saved {path.name} (sig: {sig0}-{sig1}, noise: {noise0}-{noise1})")
        

Saved SSW_001_20170225_010000Z.flac (sig: 70.6-75.6, noise: 64.6-69.6)
Saved SSW_002_20170225_020001Z.flac (sig: 787.7-792.7, noise: 781.7-786.7)
Saved SSW_003_20170225_030002Z.flac (sig: 258.4-263.4, noise: 252.39999999999998-257.4)
Saved SSW_004_20170225_050004Z.flac (sig: 31.1-36.1, noise: 25.1-30.1)
Skip SSW_005_20170225_070005Z.flac (no picked)
Skip SSW_006_20170225_100007Z.flac (no picked)
Saved SSW_007_20170225_110009Z.flac (sig: 2399.5-2404.5, noise: 2411.4-2416.4)
Saved SSW_008_20170225_120012Z.flac (sig: 336.6-341.6, noise: 330.6-335.6)
Saved SSW_009_20170225_130009Z.flac (sig: 1.4-6.4, noise: 32.9-37.9)
Saved SSW_010_20170225_140017Z.flac (sig: 226.8-231.8, noise: 220.8-225.8)


In [28]:
audio, sr = sf.read("datasets/ssw_clips/SSW_001_20170225_010000Z_signal.flac")
display(Audio(audio, rate=sr))
audio, sr = sf.read("datasets/ssw_clips/SSW_001_20170225_010000Z_noise.flac")
display(Audio(audio, rate=sr))
